In [1]:
import cv2
import mediapipe as mp
import numpy as np
import threading
from tensorflow.keras.models import load_model
from playsound import playsound



In [2]:
# Load model
model = load_model("bigger_eye_state_model.keras")



In [3]:
# Alarm control
alarm_stop_event = threading.Event()
alarm_thread = None
alarm_lock = threading.Lock()

def alarm_loop():
    alarm_stop_event.clear()
    while not alarm_stop_event.is_set():
        try:
            playsound(r'C:\Users\Sahitya\EYE_PROJECT\alarm.mp3')
        except Exception:
            break

def start_alarm():
    global alarm_thread
    with alarm_lock:
        if alarm_thread is None or not alarm_thread.is_alive():
            alarm_thread = threading.Thread(target=alarm_loop, daemon=True)
            alarm_thread.start()

def stop_alarm():
    with alarm_lock:
        alarm_stop_event.set()



In [4]:
# Mediapipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True
)



In [5]:
# Eye landmarks
RIGHT_EYE_IDX = [33, 133, 160, 159, 158, 144, 153, 154, 155, 173]
LEFT_EYE_IDX = [362, 263, 387, 386, 385, 373, 380, 381, 382, 384]

# Parameters
threshold_frames = 20
closed_frames = 0
FLIP_LEFT_BEFORE_PRED = True  # fix for left eye predictions



In [6]:
cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        eyes_closed_in_frame = False

        if result.multi_face_landmarks:
            h, w, _ = frame.shape
            predictions = []

            for eye_indices, is_left in [(RIGHT_EYE_IDX, False), (LEFT_EYE_IDX, True)]:
                eye_pts = [
                    (int(face_landmarks.landmark[i].x * w), int(face_landmarks.landmark[i].y * h))
                    for face_landmarks in result.multi_face_landmarks
                    for i in eye_indices
                ]

                if len(eye_pts) == 0:
                    continue

                x_coords = [pt[0] for pt in eye_pts]
                y_coords = [pt[1] for pt in eye_pts]
                x_min, x_max = max(0, min(x_coords)), min(w - 1, max(x_coords))
                y_min, y_max = max(0, min(y_coords)), min(h - 1, max(y_coords))

                pad = 5
                x_min = max(0, x_min - pad)
                y_min = max(0, y_min - pad)
                x_max = min(w - 1, x_max + pad)
                y_max = min(h - 1, y_max + pad)

                if x_max <= x_min or y_max <= y_min:
                    continue

                eye_img = frame[y_min:y_max, x_min:x_max]
                if eye_img.size == 0:
                    continue

                gray_eye = cv2.cvtColor(eye_img, cv2.COLOR_BGR2GRAY)
                if is_left and FLIP_LEFT_BEFORE_PRED:
                    gray_eye = cv2.flip(gray_eye, 1)  # horizontal flip

                eye_resized = cv2.resize(gray_eye, (64, 64))
                eye_normalized = eye_resized / 255.0
                eye_input = eye_normalized.reshape(1, 64, 64, 1)

                pred = model.predict(eye_input, verbose=0)[0][0]
                predictions.append(pred)

                label = "Open" if pred > 0.5 else "Closed"
                color = (0, 255, 0) if pred > 0.5 else (0, 0, 255)

                cv2.putText(frame, f"{'Left' if is_left else 'Right'}: {label} ({pred:.2f})",
                            (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)

            if predictions:
                avg_pred = np.mean(predictions)
                if avg_pred <= 0.5:
                    eyes_closed_in_frame = True

        # Counter + alarm control
        if eyes_closed_in_frame:
            closed_frames += 1
            if closed_frames >= threshold_frames:
                start_alarm()
        else:
            closed_frames = 0
            stop_alarm()

        cv2.putText(frame, f"Closed frames: {closed_frames}/{threshold_frames}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        cv2.imshow("Eye State Monitor", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    stop_alarm()
    cap.release()
    cv2.destroyAllWindows()


In [27]:
cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = face_mesh.process(rgb)

        eyes_closed_in_frame = False

        if result.multi_face_landmarks:
            h, w, _ = frame.shape
            predictions = []

            # Process each detected face
            for face_landmarks in result.multi_face_landmarks:
                for eye_indices, is_left in [(RIGHT_EYE_IDX, False), (LEFT_EYE_IDX, True)]:
                    
                    # Extract eye landmarks
                    eye_pts = [
                        (int(face_landmarks.landmark[i].x * w), 
                         int(face_landmarks.landmark[i].y * h))
                        for i in eye_indices
                    ]

                    if len(eye_pts) == 0:
                        continue

                    # Bounding box for eye
                    x_coords = [pt[0] for pt in eye_pts]
                    y_coords = [pt[1] for pt in eye_pts]
                    x_min, x_max = max(0, min(x_coords)), min(w - 1, max(x_coords))
                    y_min, y_max = max(0, min(y_coords)), min(h - 1, max(y_coords))

                    pad = 5
                    x_min = max(0, x_min - pad)
                    y_min = max(0, y_min - pad)
                    x_max = min(w - 1, x_max + pad)
                    y_max = min(h - 1, y_max + pad)

                    if x_max <= x_min or y_max <= y_min:
                        continue

                    # Crop eye
                    eye_img = frame[y_min:y_max, x_min:x_max]
                    if eye_img.size == 0:
                        continue

                    gray_eye = cv2.cvtColor(eye_img, cv2.COLOR_BGR2GRAY)

                    # Flip left eye if needed
                    if is_left and FLIP_LEFT_BEFORE_PRED:
                        gray_eye = cv2.flip(gray_eye, 1)

                    # Debug window: see cropped eyes
                    if is_left:
                        cv2.imshow("Left Eye Crop", gray_eye)
                    else:
                        cv2.imshow("Right Eye Crop", gray_eye)

                    # Resize + normalize
                    eye_resized = cv2.resize(gray_eye, (64, 64))
                    eye_normalized = eye_resized / 255.0
                    eye_input = eye_normalized.reshape(1, 64, 64, 1)

                    # Prediction
                    pred = model.predict(eye_input, verbose=0)[0][0]
                    predictions.append(pred)

                    # Label
                    label = "Open" if pred > 0.5 else "Closed"
                    color = (0, 255, 0) if pred > 0.5 else (0, 0, 255)

                    cv2.putText(frame, f"{'Left' if is_left else 'Right'}: {label} ({pred:.2f})",
                                (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
                    cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), color, 2)

            # Decision: average prediction across both eyes
            if predictions:
                avg_pred = np.mean(predictions)
                if avg_pred <= 0.5:
                    eyes_closed_in_frame = True

        # Counter + alarm
        if eyes_closed_in_frame:
            closed_frames += 1
            if closed_frames >= threshold_frames:
                start_alarm()
        else:
            closed_frames = 0
            stop_alarm()

        cv2.putText(frame, f"Closed frames: {closed_frames}/{threshold_frames}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

        cv2.imshow("Eye State Monitor", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    stop_alarm()
    cap.release()
    cv2.destroyAllWindows()